In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

# Load cleaned data
secom = pd.read_csv('../data/processed/secom_clean.csv')
steel = pd.read_csv('../data/processed/steel_clean.csv')

# ===== SECOM Feature Selection =====
X_secom = secom.drop('label', axis=1)
y_secom = secom['label']

selector = SelectKBest(f_classif, k=100)
X_secom_selected = selector.fit_transform(X_secom, y_secom)

scaler = StandardScaler()
X_secom_scaled = scaler.fit_transform(X_secom_selected)

smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_secom_scaled, y_secom)

print(f"SECOM - Before SMOTE: {X_secom.shape}")
print(f"SECOM - After SMOTE: {X_resampled.shape}")
print(f"Class balance: {dict(zip(*np.unique(y_resampled, return_counts=True)))}")

# ===== Steel Feature Engineering =====
steel['date'] = pd.to_datetime(steel['date'])
steel['hour'] = steel['date'].dt.hour
steel['day_of_week'] = steel['date'].dt.dayofweek
steel['month'] = steel['date'].dt.month
steel['is_peak_hour'] = steel['hour'].apply(lambda x: 1 if 8 <= x <= 18 else 0)
steel['usage_lag1'] = steel['Usage_kWh'].shift(1)
steel['usage_lag4'] = steel['Usage_kWh'].shift(4)
steel['usage_rolling_mean'] = steel['Usage_kWh'].rolling(window=4).mean()
steel['usage_rolling_std'] = steel['Usage_kWh'].rolling(window=4).std()

steel_fe = steel.dropna()

feature_cols = ['Lagging_Current_Reactive.Power_kVarh',
                'Leading_Current_Reactive_Power_kVarh',
                'Lagging_Current_Power_Factor',
                'Leading_Current_Power_Factor',
                'NSM', 'hour', 'day_of_week', 'month',
                'is_peak_hour', 'is_weekend', 'Load_Type_encoded',
                'usage_lag1', 'usage_lag4',
                'usage_rolling_mean', 'usage_rolling_std']

X_steel = steel_fe[feature_cols]
y_steel = steel_fe['Usage_kWh']

# Save
np.save('../data/processed/X_secom.npy', X_resampled)
np.save('../data/processed/y_secom.npy', y_resampled)
X_steel.to_csv('../data/processed/X_steel.csv', index=False)
y_steel.to_csv('../data/processed/y_steel.csv', index=False)

print(f"\nSteel features: {X_steel.shape}")
print("All data saved!")

SECOM - Before SMOTE: (1567, 562)
SECOM - After SMOTE: (2926, 100)
Class balance: {0: 1463, 1: 1463}

Steel features: (35036, 15)
All data saved!
